In [7]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
if os.environ['OPENAI_API_KEY']:
      print("API Key is set. ")

API Key is set. 


In [5]:
from langchain_openai import ChatOpenAI

c:\Users\Siddhantkumars\RAG Model\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
llm = ChatOpenAI(
    model="ai/smollm2",
    base_url="http://localhost:12434/engines/llama.cpp/v1",
    api_key="dummy",
    temperature=0
)

In [10]:
response = llm.invoke("What is AI? Tell me in one line")

print(response.content)

AI stands for Artificial Intelligence, a branch of computer science that enables machines to perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making.


## **RAG IMLEMENTATION WITH OUR OWN TEXT DATA** 

**STEP 1 : Preparing Document for our Text**

In [ ]:
from langchain_core.documents import Document

my_text = """Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]

High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.

The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers have used techniques including state space search and mathematical optimization, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind and Meta, aim to create artificial general intelligence (AGI) – AI that can complete virtually any cognitive task at least as well as a human."""

docs = [Document(page_content=my_text,metadata={"source":"ABC","documentID":"doc1"})]

docs

[Document(metadata={'source': 'ABC', 'documentID': 'doc1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]\n\nHigh-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.\n\nThe traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language process

**STEP 2 : Splitting the Document into Chunks**

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
  chunk_size = 500,
  chunk_overlap = 50)

chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'ABC', 'documentID': 'doc1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]'),
 Document(metadata={'source': 'ABC', 'documentID': 'doc1'}, page_content='High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.'),
 Document(metadata={'source': 'ABC', 'documentID': '

**STEP 3 : Create Embedding for these chunks**

In [13]:
!uv pip install langchain-huggingface sentence-transformers

Checked 2 packages in 388ms


In [14]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4667.13it/s]


**Step 4: Create Embeddings for Chunks**

In [15]:
from langchain_community.vectorstores import Chroma

vectorStore = Chroma.from_documents(
   documents=chunks,
   embedding=embedding_model
)

C:\Users\Siddhantkumars\AppData\Local\Temp\ipykernel_16184\2145515899.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [ ]:
vectors = []

for doc in chunks:
    embedding_model.embed_documents(doc.page_content)
    vectors.append(vectors)   

[[...], [...], [...], [...]]


**STEP 5: Semantics Search**

In [18]:
vectorStore.similarity_search('What is AI ? Brief in detail',k=4)

[Document(metadata={'documentID': 'doc1', 'source': 'ABC'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]'),
 Document(metadata={'documentID': 'doc1', 'source': 'ABC'}, page_content='linguistics, philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind and Meta, aim to create artificial general intelligence (AGI) – AI that can complete virtually any cognitive task at least as well as a human.'),
 Document(metadata={'documentID': 'doc1', 'source': 'ABC'}, page_content='The traditional goals of 

**Talk to LLM**

In [19]:
context = vectorStore.similarity_search("What is AI ?",k=3)

In [20]:
llm.invoke(f"What is AI ? You can answer this using the following context : {context}")

AIMessage(content='AI is a branch of computer science that deals with the development of computer systems that can perform tasks that typically require human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. These systems can be used to automate various tasks, such as image recognition, speech recognition, and natural language processing. AI has the potential to revolutionize many industries, including healthcare, finance, and transportation.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 332, 'total_tokens': 415, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'model.gguf', 'system_fingerprint': 'b1-e365e65', 'id': 'chatcmpl-dog1vChclkUTESrUE2Q9MCsTbXesmYWA', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ea60c-292a-7840-bd8c-8852467958c9-0', tool_calls=[], invalid_